In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.grid_detection import find_grid_corners, warp_grid
from src.cell_extraction import split_cells
from src.backtracking_solver import solve

In [ ]:
model = tf.keras.models.load_model('../models/digit_classifier.keras')
class_names = ['1', '2', '3', '4', '5', '6', '7', '8', '9', 'empty']  # confirm this matches your actual train_ds.class_names order

In [ ]:
img = cv2.imread('../data/raw/sample1.png')

corners = find_grid_corners(img)
warped, M = warp_grid(img, corners)
warped_gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)

cells = split_cells(warped_gray)

plt.imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
plt.title('Warped grid')
plt.axis('off')

In [ ]:
def predict_cell(cell_img, model, class_names, size=28):
    _, thresh_cell = cv2.threshold(cell_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    resized = cv2.resize(thresh_cell, (size, size))
    normalized = resized.astype('float32') / 255.0
    input_tensor = normalized.reshape(1, size, size, 1)
    pred = model.predict(input_tensor, verbose=0)
    class_idx = np.argmax(pred)
    confidence = np.max(pred)
    label = class_names[class_idx]
    return label, confidence

grid = []
confidences = []

for row in range(9):
    grid_row = []
    conf_row = []
    for col in range(9):
        cell = cells[row * 9 + col]
        label, conf = predict_cell(cell, model, class_names)
        digit = 0 if label == 'empty' else int(label)
        grid_row.append(digit)
        conf_row.append(conf)
    grid.append(grid_row)
    confidences.append(conf_row)

print("Detected grid:")
for row in grid:
    print(row)
print("Confidences grid:")
for row in confidences:
    print(row)


In [ ]:
threshold = 0.7
print("Low-confidence cells (row, col, label, confidence):")
for row in range(9):
    for col in range(9):
        if confidences[row][col] < threshold:
            print(f"({row},{col}) → {grid[row][col]}, conf={confidences[row][col]:.2f}")

In [ ]:
import copy
solved_grid = copy.deepcopy(grid)

if solve(solved_grid):
    print("Solved:")
    for row in solved_grid:
        print(row)
else:
    print("Could not solve — likely a misread digit somewhere.")

In [ ]:
def overlay_solution(warped_img, original_grid, solved_grid, cell_size=100):
    overlay = warped_img.copy()
    for row in range(9):
        for col in range(9):
            if original_grid[row][col] == 0:  # only draw digits that were originally empty
                digit = solved_grid[row][col]
                x = col * cell_size + cell_size // 3
                y = row * cell_size + int(cell_size * 0.7)
                cv2.putText(overlay, str(digit), (x, y),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 128, 0), 2, cv2.LINE_AA)
    return overlay

warped_size = warped.shape[0]
cell_size = warped_size // 9

result = overlay_solution(warped, grid, solved_grid, cell_size)

plt.figure(figsize=(6,6))
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.title('Solved overlay')
plt.axis('off')

In [ ]:
def unwarp_overlay(original_img, overlay_img, M):
    h, w = original_img.shape[:2]
    M_inv = np.linalg.inv(M)
    unwarped_overlay = cv2.warpPerspective(overlay_img, M_inv, (w, h))

    mask = cv2.warpPerspective(np.ones_like(overlay_img) * 255, M_inv, (w, h))
    mask_gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
    _, mask_binary = cv2.threshold(mask_gray, 1, 255, cv2.THRESH_BINARY)

    final = original_img.copy()
    final[mask_binary > 0] = unwarped_overlay[mask_binary > 0]
    return final

final_result = unwarp_overlay(img, result, M)

plt.figure(figsize=(6,6))
plt.imshow(cv2.cvtColor(final_result, cv2.COLOR_BGR2RGB))
plt.title('Final: solved digits overlaid on original photo')
plt.axis('off')